# Dispatch Viewer

In [ ]:
from ipywidgets import Dropdown, Select
from upath import UPath
from src import runner
from ipyfilechooser import FileChooser
import xlwings as xw
from loguru import logger
from tqdm.notebook import trange, tqdm
import pandas as pd

In [ ]:
genx_wb = FileChooser(default_path=".", default_filename="Kentucky Load Resource Model.xlsb", title="Connect to a GenX spreadsheet: ", filter_pattern="*.xls*", show_hidden=False)
genx_wb

In [ ]:
selected_case = Select(
    options=[p for p in runner.get_solved_cases(UPath("./cases")) if "pcm" in str(p) and (p / "results").exists()],
    description="Available cases: ",
    rows=10,
    layout=dict(width="max-content"),
    style=dict(description_width="max-content"),
)
selected_case

In [ ]:
wb = xw.Book(UPath(genx_wb.value))

type_map = wb.sheets["GenX Resources"].range("TypeMap").options(pd.DataFrame, index=0).value.dropna().set_index("resource").squeeze(axis=1).to_dict()

base_folder = UPath(selected_case.value) / "results" / "results_p1"

df = pd.read_csv(base_folder / "power.csv").iloc[2:, 1:-1]
df = df.reset_index(drop=True)
df.index = pd.Timestamp("1/1/2007") + pd.to_timedelta(df.index, unit="h")
df = df.T.groupby(type_map).sum().T
df = df[wb.sheets["List"].range("Clusters").options(list).value]
df = df[[col for col in df.columns if df[col].sum()>0]]

load = -pd.read_csv(base_folder / "power_balance.csv", skiprows=[1, 2], index_col=0).iloc[:, -1].squeeze()
load.index = df.index

import plotly.graph_objects as go

# Hourly
fig = go.Figure(
    data=[
        go.Scatter(x=df.index, y=df[col], fill="tonexty", fillcolor=runner.color_map[col], stackgroup=1, name=col)
        for col in df
    ]
)
fig.update_traces(line_width=0)
fig.add_trace(go.Scatter(x=df.index, y=load, mode="lines", name=load.name, stackgroup=None, line_color="black"))
fig.show()

## Month-hour
month_hour = df.groupby(by=[df.index.month, df.index.hour]).mean()
month_hour = month_hour.reset_index(names=["month", "hour"])

# blanks = pd.DataFrame(index=[(24 * n) + 0.5 for n in range(1, 12)])
# blanks["month"] = blanks.index // 24
# blanks["hour"] = 24.5

# month_hour = pd.concat([month_hour, blanks], axis=0).sort_index()

mh_fig = go.Figure(
    data=[
        go.Scatter(x=[month_hour["month"], month_hour["hour"]], y=month_hour[col], fill="tonexty", fillcolor=runner.color_map[col], stackgroup=1, name=col, connectgaps=False)
        for col in month_hour if col not in ["month", "hour"]
    ]
)
for i in range(1, 12):
    mh_fig.add_vrect(x0=i*24 - 1, x1=i*24, line_width=0, fillcolor="white")

mh_fig.update_traces(line_width=0)
mh_load = load.groupby(by=[load.index.month, load.index.hour]).mean()
mh_load = mh_load.to_frame().reset_index(names=["month", "hour"])
mh_fig.add_trace(go.Scatter(x=[mh_load["month"], mh_load["hour"]], y=mh_load["Demand"], name="Demand", stackgroup=None, mode="lines", line_color="black"))

mh_fig.update_layout(title=f"<b>{selected_case.value.stem}</b><br>Month-Hour Average Dispatch")
# mh_fig.update_xaxes(tickson="labels")
mh_fig.show()